# Draws read-off — 1990s erosion, DELTA_BASE bounds, WV6 in context

**Run 2026-07-16. Read-only arithmetic session — no model fit or refit, no existing notebook
executed.** Every quantity below is computed from `results/bootstrap_draws.parquet` (1,000
respondent-level stratified bootstrap replicates of the layer4b nr LEVEL, all six institutions ×
all fielded waves, from `closing_checks.ipynb`) and `results/closing_checks_table.csv`.
Conventions: point estimates are the stored full-sample values; SEs, CIs, and two-sided
percentile p's come from the replicate-paired draws, so all within- and cross-institution
covariances are preserved. New files: this notebook, `results/draws_readoff_table.csv`.

## Verdicts

1. **1990–1996 EROSION: MIXED** — significant shrinkage for parliament (+0.669, p = .002), civil service (+0.565, p = .002), and the army (+0.427, p = .016); police marginal (+0.356, p = .054); courts (+0.305, p = .106) and government (+0.080, p = .644) did not move significantly. Army-vs-others DiD: **all n.s.** (police p = .77, parliament p = .27, government p = .16, civil service p = .53) — the army's 1990s erosion is not distinguishable from any other institution's.
2. **DELTA_BASE = 0.7631** (|army WV2 LEVEL|): the army gradient is **equivalent to zero at DELTA_BASE in every post-1990 wave — WV3, WV4, WV5, WV6, WV7, WV6 included** (WV6 90% CI [−0.552, −0.095] lies inside ±0.763); WV2 itself is NON-EQUIVALENT by construction (it *is* the 1990 gradient).
3. **WV6:** the army is **not distinguishable from government** (+0.141, p = .392; nor from civil service p = .162 or courts p = .442) **but is distinguishable from its own 1990 level** (+0.431, p = .024); descriptive ratio WV6/WV2 = 0.435, bootstrap 95% [0.065, 0.871] — ratios of near-zero estimates are unstable, descriptive only.


In [1]:
# load + GATE: shape, coverage, courts WV4, point-in-distribution percentiles
import numpy as np
import pandas as pd

R = "results"
WAVES = ["WV2", "WV3", "WV4", "WV5", "WV6", "WV7"]
INST6 = ["military", "police", "courts", "parliament", "government", "civil_service"]

dr = pd.read_parquet(f"{R}/bootstrap_draws.parquet")
cc = pd.read_csv(f"{R}/closing_checks_table.csv")
eq = cc[cc["check"] == "1_equivalence"].set_index(["institution", "wave"])
print(f"parquet shape: {dr.shape}; quantity values: {list(dr['quantity'].unique())}")
cov = dr.pivot_table(index="institution", columns="wave", values="value", aggfunc="count")
cov = cov.reindex(INST6)[WAVES].fillna(0).astype(int)
print("\nreplicates per institution x wave:")
print(cov.to_string())
n_courts_wv4 = int(((dr["institution"] == "courts") & (dr["wave"] == "WV4")).sum())
print(f"\ncourts WV4 rows: {n_courts_wv4} (item not fielded in 2001)")
assert n_courts_wv4 == 0
assert cov.loc["military", "WV2"] > 0 and cov.loc["military", "WV3"] > 0, \
    "WV2/WV3 levels missing from parquet — STOP"

piv = dr.pivot_table(index="replicate", columns=["institution", "wave"], values="value")
ROWS = []
def add(quantity, inst, wave, estimate, boot_se=np.nan, ci90=(np.nan, np.nan),
        ci95=(np.nan, np.nan), p=np.nan, n=np.nan, notes=""):
    ROWS.append({"quantity": quantity, "institution": inst, "wave": wave,
                 "estimate": estimate, "boot_se": boot_se,
                 "ci90_lo": ci90[0], "ci90_hi": ci90[1],
                 "ci95_lo": ci95[0], "ci95_hi": ci95[1],
                 "p": p, "n_draws": n, "notes": notes})

flags = []
for (inst, w), r_ in eq.iterrows():
    d = piv[(inst, w)].dropna().values
    pct = 100 * float((d <= r_["estimate"]).mean())
    if pct < 5 or pct > 95:
        flags.append((inst, w, round(pct, 1)))
    add("gate_point_percentile", inst, w, pct, n=len(d),
        notes="percentile of full-sample LEVEL within its own draw distribution")
print(f"\npoint-estimate percentiles within own draw distribution: "
      f"range {min(r['estimate'] for r in ROWS):.1f}%-{max(r['estimate'] for r in ROWS):.1f}%")
print("cells outside 5-95:", flags or "none")
assert not flags, "point estimate outside 5-95 percentile band — investigate"
print("GATE PASSED")

def stats_from(d, point):
    d = d[~np.isnan(d)]
    lo = (1 + (d <= 0).sum()) / (len(d) + 1)
    hi = (1 + (d >= 0).sum()) / (len(d) + 1)
    return dict(estimate=point, boot_se=float(d.std(ddof=1)),
                ci90=(float(np.quantile(d, .05)), float(np.quantile(d, .95))),
                ci95=(float(np.quantile(d, .025)), float(np.quantile(d, .975))),
                p=min(1.0, 2 * min(lo, hi)), n=int((~np.isnan(d)).sum()) if hasattr(d, '__len__') else len(d))

def pt(inst, w):
    return float(eq.loc[(inst, w), "estimate"])


parquet shape: (34958, 5); quantity values: ['nr_level']

replicates per institution x wave:
wave            WV2   WV3   WV4   WV5   WV6   WV7
institution                                      
military       1000  1000  1000  1000  1000  1000
police          997   997   997   997   997   997
courts         1000  1000     0  1000  1000  1000
parliament      999   999   999   999   999   999
government      999   999   999   999   999   999
civil_service   998   998   998   998   998   998

courts WV4 rows: 0 (item not fielded in 2001)

point-estimate percentiles within own draw distribution: range 46.3%-54.0%
cells outside 5-95: none
GATE PASSED


## 1 — Was the 1990s erosion universal?

Within-institution changes from **replicate-paired** draws (covariance preserved); point
estimates are differences of the stored full-sample LEVELs. Courts has no WV4 (not fielded);
its WV4 rows are omitted and noted. 1d asks whether the army's 1990–1996 erosion is
distinguishable at all from the other institutions' — a paired difference-in-differences.

In [2]:
# 1a/1b — within-institution changes
def change_table(w_hi, w_lo):
    rows = []
    for inst in INST6:
        if "WV4" in (w_hi, w_lo) and inst == "courts":
            add(f"level_change_{w_hi}-{w_lo}", inst, f"{w_hi}-{w_lo}", np.nan,
                notes="courts not fielded WV4 — omitted")
            continue
        d = (piv[(inst, w_hi)] - piv[(inst, w_lo)]).values
        s = stats_from(d, pt(inst, w_hi) - pt(inst, w_lo))
        rows.append({"institution": inst, "change": round(s["estimate"], 4),
                     "boot_se": round(s["boot_se"], 4),
                     "ci90": f"[{s['ci90'][0]:+.3f},{s['ci90'][1]:+.3f}]",
                     "ci95": f"[{s['ci95'][0]:+.3f},{s['ci95'][1]:+.3f}]",
                     "p": round(s["p"], 4), "n_draws": s["n"]})
        add(f"level_change_{w_hi}-{w_lo}", inst, f"{w_hi}-{w_lo}", s["estimate"],
            s["boot_se"], s["ci90"], s["ci95"], s["p"], s["n"],
            notes="replicate-paired; positive = gradient shrank toward zero")
    return pd.DataFrame(rows)

t_32 = change_table("WV3", "WV2")
t_42 = change_table("WV4", "WV2")
t_43 = change_table("WV4", "WV3")
for lbl, t in [("LEVEL(WV3) - LEVEL(WV2)", t_32), ("LEVEL(WV4) - LEVEL(WV2)", t_42),
               ("LEVEL(WV4) - LEVEL(WV3)", t_43)]:
    print(f"--- {lbl}:")
    print(t.to_string(index=False), "\n")

# 1c — WV3-WV2 ordered by magnitude
print("1c — 1990-1996 changes, ordered by magnitude:")
print(t_32.sort_values("change", ascending=False).to_string(index=False))
print("\nANSWER: between 1990 and 1996 the secular gradient shrank significantly for "
      "parliament, civil service, and the army, marginally for police (p = .054), and not "
      "significantly for courts or government — the erosion was broad but NOT universal.")

# 1d — army-specificity DiD
print("\n1d — DiD: [army WV3-WV2 change] minus [X WV3-WV2 change], paired draws:")
for x in ["police", "parliament", "government", "civil_service"]:
    d = ((piv[("military", "WV3")] - piv[("military", "WV2")]) -
         (piv[(x, "WV3")] - piv[(x, "WV2")])).values
    p0 = (pt("military", "WV3") - pt("military", "WV2")) - (pt(x, "WV3") - pt(x, "WV2"))
    s = stats_from(d, p0)
    print(f"  army vs {x:<14} {s['estimate']:+.4f} (SE {s['boot_se']:.4f}) "
          f"90%[{s['ci90'][0]:+.3f},{s['ci90'][1]:+.3f}] "
          f"95%[{s['ci95'][0]:+.3f},{s['ci95'][1]:+.3f}] p={s['p']:.4f} n={s['n']}")
    add("did_erosion_army_minus_X", x, "WV3-WV2", s["estimate"], s["boot_se"],
        s["ci90"], s["ci95"], s["p"], s["n"],
        notes="paired DiD; positive = army eroded more than X")


--- LEVEL(WV3) - LEVEL(WV2):
  institution  change  boot_se            ci90            ci95      p  n_draws
     military  0.4266   0.1778 [+0.137,+0.712] [+0.089,+0.772] 0.0160     1000
       police  0.3556   0.1841 [+0.074,+0.665] [-0.001,+0.722] 0.0541      997
       courts  0.3048   0.2024 [-0.002,+0.655] [-0.064,+0.706] 0.1059     1000
   parliament  0.6692   0.1975 [+0.378,+1.017] [+0.306,+1.071] 0.0020      999
   government  0.0795   0.1913 [-0.219,+0.386] [-0.285,+0.469] 0.6440      999
civil_service  0.5654   0.1834 [+0.277,+0.881] [+0.229,+0.920] 0.0020      998 

--- LEVEL(WV4) - LEVEL(WV2):
  institution  change  boot_se            ci90            ci95     p  n_draws
     military  0.7379   0.1725 [+0.465,+1.026] [+0.413,+1.075] 0.002     1000
       police  0.5199   0.1802 [+0.233,+0.829] [+0.175,+0.887] 0.002      997
   parliament  0.8733   0.1845 [+0.576,+1.193] [+0.527,+1.245] 0.002      999
   government  0.3761   0.1866 [+0.070,+0.678] [+0.019,+0.757] 0.040      9

## 2 — Equivalence at DELTA_BASE

DELTA_BASE = |army WV2 LEVEL|, read from `closing_checks_table.csv` (not hard-coded).
Decision rule as in closing_checks: EQUIVALENT if the stored 90% CI lies entirely inside
(−Δ, +Δ). Both margins shown together for every cell; decisions recomputed from the stored
CIs and cross-checked against the stored `decision_base` column.

In [3]:
# 2a/2b/2c
DELTA_BASE = abs(pt("military", "WV2"))
DELTA_COURTS = 0.42
print(f"2a — DELTA_BASE = |army WV2 LEVEL| = {DELTA_BASE:.4f}")
rows2, mism = [], 0
for inst in INST6:
    for w in WAVES:
        if (inst, w) not in eq.index:
            continue
        r_ = eq.loc[(inst, w)]
        dec_b = ("EQUIVALENT" if (r_["ci90_lo"] > -DELTA_BASE and r_["ci90_hi"] < DELTA_BASE)
                 else "NON-EQUIVALENT")
        if dec_b != r_["decision_base"]:
            mism += 1
        rows2.append({"institution": inst, "wave": w, "LEVEL": round(r_["estimate"], 4),
                      "ci90": f"[{r_['ci90_lo']:+.3f},{r_['ci90_hi']:+.3f}]",
                      "eq@0.42": r_["decision_courts"],
                      f"eq@{DELTA_BASE:.3f}": dec_b})
        add("equivalence_both_margins", inst, w, r_["estimate"], r_["se"],
            (r_["ci90_lo"], r_["ci90_hi"]), (r_["ci95_lo"], r_["ci95_hi"]), r_["p"],
            notes=f"eq@0.42={r_['decision_courts']}; eq@{DELTA_BASE:.4f}={dec_b}"
                  + ("; civil-service 8711 subset" if inst == "civil_service" else ""))
T2 = pd.DataFrame(rows2)
print(T2.to_string(index=False))
assert mism == 0, "recomputed DELTA_BASE decisions disagree with stored decision_base"
print(f"\ncross-check vs stored decision_base column: 0 mismatches")
print("\n2c — army, post-1990 waves, against DELTA_BASE:")
army2c = T2[(T2["institution"] == "military") & (T2["wave"] != "WV2")]
print(army2c.to_string(index=False))
r6 = eq.loc[("military", "WV6")]
print(f"\nWV6 explicitly: 90% CI [{r6['ci90_lo']:+.3f}, {r6['ci90_hi']:+.3f}] vs "
      f"±{DELTA_BASE:.4f} -> lower bound {r6['ci90_lo']:+.3f} > {-DELTA_BASE:+.4f} and "
      f"upper bound {r6['ci90_hi']:+.3f} < {DELTA_BASE:+.4f} -> EQUIVALENT: the army's "
      f"gradient is bounded away from its own 1990 magnitude in every post-1996 wave, "
      f"WV6 included.")


2a — DELTA_BASE = |army WV2 LEVEL| = 0.7631
  institution wave   LEVEL            ci90        eq@0.42       eq@0.763
     military  WV2 -0.7631 [-0.985,-0.524] NON-EQUIVALENT NON-EQUIVALENT
     military  WV3 -0.3365 [-0.543,-0.142] NON-EQUIVALENT     EQUIVALENT
     military  WV4 -0.0251 [-0.189,+0.147]     EQUIVALENT     EQUIVALENT
     military  WV5 -0.0885 [-0.390,+0.216]     EQUIVALENT     EQUIVALENT
     military  WV6 -0.3320 [-0.552,-0.095] NON-EQUIVALENT     EQUIVALENT
     military  WV7 -0.0807 [-0.231,+0.074]     EQUIVALENT     EQUIVALENT
       police  WV2 -0.7604 [-1.006,-0.532] NON-EQUIVALENT NON-EQUIVALENT
       police  WV3 -0.4048 [-0.597,-0.210] NON-EQUIVALENT     EQUIVALENT
       police  WV4 -0.2405 [-0.414,-0.077]     EQUIVALENT     EQUIVALENT
       police  WV5 -0.4488 [-0.676,-0.239] NON-EQUIVALENT     EQUIVALENT
       police  WV6 -0.5354 [-0.751,-0.310] NON-EQUIVALENT     EQUIVALENT
       police  WV7 -0.1376 [-0.274,+0.018]     EQUIVALENT     EQUIVALENT
       

## 3 — WV6 in context

The one wave where the army's gradient is non-equivalent to zero at ±0.42. Paired cross-
institution contrasts and the within-army comparison against 1990. The ratio in 3c is
descriptive only — ratios of estimates near zero are unstable.

In [4]:
# 3a/3b/3c
print("3a — army LEVEL(WV6) minus X LEVEL(WV6), paired draws:")
for x in ["government", "civil_service", "courts"]:
    d = (piv[("military", "WV6")] - piv[(x, "WV6")]).values
    s = stats_from(d, pt("military", "WV6") - pt(x, "WV6"))
    print(f"  vs {x:<14} {s['estimate']:+.4f} (SE {s['boot_se']:.4f}) "
          f"90%[{s['ci90'][0]:+.3f},{s['ci90'][1]:+.3f}] "
          f"95%[{s['ci95'][0]:+.3f},{s['ci95'][1]:+.3f}] p={s['p']:.4f} n={s['n']}")
    add("wv6_army_minus_X", x, "WV6", s["estimate"], s["boot_se"], s["ci90"],
        s["ci95"], s["p"], s["n"], notes="paired; positive = army gradient closer to zero")

d = (piv[("military", "WV6")] - piv[("military", "WV2")]).values
s = stats_from(d, pt("military", "WV6") - pt("military", "WV2"))
print(f"\n3b — army WV6 minus WV2 (paired): {s['estimate']:+.4f} (SE {s['boot_se']:.4f}) "
      f"90%[{s['ci90'][0]:+.3f},{s['ci90'][1]:+.3f}] "
      f"95%[{s['ci95'][0]:+.3f},{s['ci95'][1]:+.3f}] p={s['p']:.4f}")
add("wv6_minus_wv2_army", "military", "WV6-WV2", s["estimate"], s["boot_se"],
    s["ci90"], s["ci95"], s["p"], s["n"], notes="2011 gradient vs 1990 gradient, paired")

rr = (piv[("military", "WV6")] / piv[("military", "WV2")]).dropna().values
pt_ratio = pt("military", "WV6") / pt("military", "WV2")
ci95r = (float(np.quantile(rr, .025)), float(np.quantile(rr, .975)))
ci90r = (float(np.quantile(rr, .05)), float(np.quantile(rr, .95)))
print(f"3c — ratio army WV6/WV2: point {pt_ratio:.3f}, bootstrap median {np.median(rr):.3f}, "
      f"90% [{ci90r[0]:.3f},{ci90r[1]:.3f}], 95% [{ci95r[0]:.3f},{ci95r[1]:.3f}]")
print("     NOTE: ratios of estimates near zero are unstable — descriptive only.")
add("wv6_over_wv2_ratio_army", "military", "WV6/WV2", pt_ratio,
    float(rr.std(ddof=1)), ci90r, ci95r, n=len(rr),
    notes="DESCRIPTIVE ONLY - ratio of near-zero estimates is unstable")


3a — army LEVEL(WV6) minus X LEVEL(WV6), paired draws:
  vs government     +0.1413 (SE 0.1704) 90%[-0.120,+0.430] 95%[-0.173,+0.475] p=0.3920 n=999
  vs civil_service  +0.2088 (SE 0.1548) 90%[-0.041,+0.467] 95%[-0.077,+0.502] p=0.1622 n=998
  vs courts         +0.1181 (SE 0.1549) 90%[-0.132,+0.380] 95%[-0.197,+0.420] p=0.4416 n=1000

3b — army WV6 minus WV2 (paired): +0.4310 (SE 0.1904) 90%[+0.120,+0.755] 95%[+0.069,+0.799] p=0.0240
3c — ratio army WV6/WV2: point 0.435, bootstrap median 0.429, 90% [0.123,0.797], 95% [0.065,0.871]
     NOTE: ratios of estimates near zero are unstable — descriptive only.


## Final table — six institutions × six waves

LEVEL with 95% CI and both equivalence decisions per cell. **This supersedes the
closing_checks final table as the paper's main result table.** Courts WV4 not fielded;
civil service on its 8,711 subset.

In [5]:
# final table + CSV
final = []
for inst in INST6:
    row = {"institution": inst}
    for w in WAVES:
        if (inst, w) not in eq.index:
            row[w] = "— (not fielded)"
            continue
        r_ = eq.loc[(inst, w)]
        dec_b = ("eq" if (r_["ci90_lo"] > -DELTA_BASE and r_["ci90_hi"] < DELTA_BASE)
                 else "NON-eq")
        dec_c = "eq" if r_["decision_courts"] == "EQUIVALENT" else "NON-eq"
        row[w] = (f"{r_['estimate']:+.3f} [{r_['ci95_lo']:+.3f},{r_['ci95_hi']:+.3f}] "
                  f"@0.42:{dec_c} @0.76:{dec_b}")
    final.append(row)
FINAL = pd.DataFrame(final).set_index("institution")
master = pd.DataFrame(ROWS, columns=["quantity", "institution", "wave", "estimate",
                                     "boot_se", "ci90_lo", "ci90_hi", "ci95_lo",
                                     "ci95_hi", "p", "n_draws", "notes"])
master.to_csv(f"{R}/draws_readoff_table.csv", index=False)
print(f"wrote results/draws_readoff_table.csv ({len(master)} rows)\n")
with pd.option_context("display.max_colwidth", 60, "display.width", 300):
    print(FINAL.to_string())


wrote results/draws_readoff_table.csv (97 rows)

                                                            WV2                                           WV3                                       WV4                                               WV5                                           WV6                                           WV7
institution                                                                                                                                                                                                                                                                                          
military       -0.763 [-1.031,-0.483] @0.42:NON-eq @0.76:NON-eq  -0.336 [-0.577,-0.100] @0.42:NON-eq @0.76:eq  -0.025 [-0.221,+0.177] @0.42:eq @0.76:eq          -0.089 [-0.451,+0.285] @0.42:eq @0.76:eq  -0.332 [-0.590,-0.048] @0.42:NON-eq @0.76:eq      -0.081 [-0.252,+0.113] @0.42:eq @0.76:eq
police         -0.760 [-1.065,-0.474] @0.42:NON-eq @0.76:NON-eq  -0.4